In [2]:
# SoRL in modded-gpt compatible fashion (for ultra fast pre-training)
# 1. pre-training demands simple model architecture, even .generate function can be wrapped around the trained model afterwards
# 2. no need to include 'kv-cache' for the pre-training experiment here

In [ ]:
from sorl.model import CausalSelfAttention, Block, GPTConfig
import torch 

# mock input 
x = torch.randn(2, 1024, 768)

config = GPTConfig()
attn = CausalSelfAttention(dim=768, n_head=6)
block = Block(config=config)

y, v1 = attn(x)
x, v1 = block(x, v1, x, None)

In [3]:
import torch 
from sorl.gat import GATConfig, GAT

gat_config = GATConfig(vocab_sizes=[128,8],
          n_layer=12,
          n_head=6,
          n_embd=768,
          flex_kernel_options=None)

model = GAT(gat_config)


token_ids = torch.randint(0, 128 + 8, (2, 4))
idx = token_ids[:, :-1].contiguous()
target = token_ids[:, 1:].contiguous()


# forward pass ()
ppt = model(idx, target, 1024)

In [4]:
from sorl.gat import parallel_denoise
from sorl.gat import generate 


num_iterations = 5 
memory_span = 1024 
temperature = 0.0

parallel_denoise(model, idx, num_iterations=5, memory_span=1024, temperature=0.0)

generate(model, idx, max_new_tokens=5, abstraction_interval=3)


tensor([[ 42,  86, 110, 129,   0,   0, 129,   0],
        [ 46,  21,   4, 129,   0,   0, 129,   0]])

In [24]:
idx = torch.tensor([
    [4, 129, 129, 129],
    [3, 10, 129, 129]
])

In [6]:
import torch 
from sorl.gat_act import GATConfig, GAT

gat_config = GATConfig(vocab_sizes=[128,8],
          n_layer=12,
          n_head=6,
          n_embd=768,
          flex_kernel_options=None)

model = GAT(gat_config)

from sorl.gat_act import infer_level

# idx = torch.randint(0, model.vocab_sizes.sum(), (2, 4)).contiguous()
idx = torch.tensor([
    [4, 129, 130, 129],
    [3, 10, 130, 129]
]).contiguous()
levels = infer_level(idx, model.vocab_sizes)
abstract_mask = (levels > 0)

In [2]:
# Continuous recursion & Discrete recursion
# -------------------------------------------
# Remark 1. 'abstract_mask' is more of a 'recursion_mask' -- of course we only recurse on abstract tokens
# Reflection 1. We use 'recursion_mask' only for denoising process & forward pass
# 'Denoise' method basically does discrete recursion, we should be more upfront about this in terminology
# Idea 1. perhaps the explicit separation between continuous recursion & discrete recursion is okay? we could 
#         experiment both individually, and perhaps asynchronous recuring on them makes sense ...
# Reflection 2. When generate new tokens, we'd also recurse things simultaneously, we almost need a 'forward_with_recursion' method
#               that does discrete & continuous recursion simultaneously, at least we should not waste the next token prediction logits
#               (for instance, every recursion leads to a different next-token prediction)
# Reflection 3. From reflection-2, it seems that there is no real difference between 'generate' method and 'recursion' method
#               since generate is basically just 'recursion' + decode on last token representation


# it does feels like we can simplify it. 
# as what I need is a 'recursion' method, that perform some pre-determined # of continuous recursion (via passing and using abstract_repr), as well as some pre-determined # of discrete recursion (via passing and re-using the deocded abstract tokens)

# here a reflection 4. is that there is no reason why ACT is only done on the continuous recursion, we ought to try ACT on both continuous recursion and discrete recursion --- according to the lesson from TRM/HRM work, ACT on discrete recursion is probably more meaningful



# Reflection 1. 
# - continuous recursion assumes invariant recursion tokens, rep + wte = new_wte
# - discrete recursion changes recursion tokens, this makes inner-outer loop more sensible than joint loop

# Reflection 2. 
# - to mimic TRM/HRM settings, we could separate inner/outer loop
# - inner loop contains fixed # of continuous recursion
# - outer loop does ACT & discrete recursion



In [3]:
idx = idx.clone() 

import torch.nn.functional as F

levels = infer_level(idx, model.vocab_sizes)
abs_mask = levels > 0 
abs_mask[:, 0] = False # don't need to evict tokens during generation, we have sparse attention
recursion_mask = abs_mask.clone()

from sorl.gat_act import get_logits_mask

abstract_repr = torch.zeros(abs_mask.sum(), model.n_embd, device=idx.device)

max_iterations = 5 
memory_span = 1024 
temperature = 0.0

losses = [] 
for iteration in range(max_iterations):
    # Idea. every iteration requires a loss computation, and a detach operation on the abstract_repr

    # --- forward pass & continuous recursion ---
    loss, logits, new_abs_repr, act = model.forward(idx, abstract_repr, abs_mask, memory_span)
    losses.append(loss)

    recursion_mask = recursion_mask & ~act.unsqueeze(1)
    if not recursion_mask.any(): 
        break 

    # --- discrete recursion --- 
    predict_mask = torch.roll(recursion_mask, -1, dims=1) # prev token embedding predict next token
    predict_mask[:, -1] = False
    recursion_logits = logits[predict_mask]
    
    recursion_levels = levels[recursion_mask]
    logits_mask = get_logits_mask(recursion_levels, model.vocab_sizes)
    recursion_logits = torch.where(logits_mask, recursion_logits, torch.tensor(float('-inf'), device=model.device))

    if temperature == 0.0:
        recursion_tokens = torch.argmax(recursion_logits, dim=-1)
    else:
        recursion_tokens = torch.multinomial(F.softmax(recursion_logits / temperature, dim=-1), num_samples=1).squeeze(-1)

    idx[recursion_mask] = recursion_tokens

    # --- continuous recursion --- 
    update_mask = recursion_mask[abs_mask].unsqueeze(-1)
    abstract_repr = torch.where(update_mask, new_abs_repr, abstract_repr)

total_loss = sum(losses) / len(losses)

In [7]:
def recursion(model, idx, max_iterations=5, memory_span=1024, temperature=0.0):
    """
    Perform iterative recursion with continuous and discrete updates.
    - continuous & discrete recursion 
    - ACT-based early stopping
    - loss computation at each iteration
    """
    idx = idx.clone() 
    
    # Initialize masks and representations
    levels = infer_level(idx, model.vocab_sizes)
    abs_mask = levels > 0 
    abs_mask[:, 0] = False
    recursion_mask = abs_mask.clone()
    
    abstract_repr = torch.zeros(abs_mask.sum(), model.n_embd, device=idx.device)
    
    losses = [] 
    for iteration in range(max_iterations):

        # --- forward pass ---
        loss, logits, new_abs_repr, act = model.forward(idx, abstract_repr, abs_mask, memory_span)
        losses.append(loss)
        
        # --- ACT early stop --- 
        recursion_mask = recursion_mask & ~act.unsqueeze(1)
        if not recursion_mask.any(): 
            break 
        
        # --- discrete recursion
        predict_mask = torch.roll(recursion_mask, -1, dims=1)
        predict_mask[:, -1] = False
        recursion_logits = logits[predict_mask]
        
        recursion_levels = levels[recursion_mask]
        logits_mask = get_logits_mask(recursion_levels, model.vocab_sizes)
        recursion_logits = torch.where(logits_mask, recursion_logits, 
                                      torch.tensor(float('-inf'), device=model.device))
        
        if temperature == 0.0:
            recursion_tokens = torch.argmax(recursion_logits, dim=-1)
        else:
            recursion_tokens = torch.multinomial(F.softmax(recursion_logits / temperature, dim=-1), 
                                                num_samples=1).squeeze(-1)
        
        idx[recursion_mask] = recursion_tokens
        
        # --- continuous recursion --- 
        update_mask = recursion_mask[abs_mask].unsqueeze(-1)
        abstract_repr = torch.where(update_mask, new_abs_repr, abstract_repr)
    
    total_loss = sum(losses) / len(losses) if losses else torch.tensor(0.0)
    
    return idx, abstract_repr, total_loss

In [8]:
recursion(model, idx, max_iterations=5, memory_span=1024, temperature=0.0)

(tensor([[  4, 129, 129, 129],
         [  3,  10, 129, 129]]),
 tensor([[-0.0119,  1.0562,  1.8032,  ...,  0.4582, -0.4744, -0.8242],
         [ 0.0909,  1.0673,  1.7488,  ...,  0.4203, -0.4848, -0.7953],
         [-0.0119,  1.0562,  1.8032,  ...,  0.4582, -0.4744, -0.8242],
         [ 0.0909,  1.0673,  1.7488,  ...,  0.4203, -0.4848, -0.7953],
         [-0.0119,  1.0562,  1.8032,  ...,  0.4582, -0.4744, -0.8242]]),
 tensor([4.9127, 4.9127, 4.9127, 4.9127, 4.9127, 4.9127],
        grad_fn=<DivBackward0>))